# 03 マクロデータ・パイプライン(point-in-time)

プラットフォームの核となるリゴール:**マクロ指標は release_date 前に使えない**。改定(revision)があるとき、`as_of(過去日)` は当時の初出値を返し、`latest()` は最新改定値を返す。**前埋めしない**。まず合成データで保証を実演(オフラインでも必ず動く)し、鍵があれば FRED/ALFRED の実 vintage でも確認する。

## 実演: 改定される CPI(合成 vintage、ネット不要)

In [1]:
import pandas as pd
from irp.macro.schema import MacroObservation, to_macro_frame
from irp.macro.store import as_of, latest, revisions

obs = [
    # 1月分: 初出 02-15 = 3.0 -> 改定 03-15 = 3.2
    MacroObservation('cpi','US',pd.Timestamp('2024-01-01'),pd.Timestamp('2024-01-31'),
                     pd.Timestamp('2024-02-15'),3.0,'ALFRED',vintage_available=True),
    MacroObservation('cpi','US',pd.Timestamp('2024-01-01'),pd.Timestamp('2024-01-31'),
                     pd.Timestamp('2024-03-15'),3.2,'ALFRED',vintage_available=True),
    # 2月分: 初出 03-15 = 2.8
    MacroObservation('cpi','US',pd.Timestamp('2024-02-01'),pd.Timestamp('2024-02-29'),
                     pd.Timestamp('2024-03-15'),2.8,'ALFRED',vintage_available=True),
]
frame = to_macro_frame(obs)
frame[['period_start','release_date','value']]

,period_start,release_date,value
0,2024-01-01,2024-02-15,3.0
1,2024-01-01,2024-03-15,3.2
2,2024-02-01,2024-03-15,2.8


In [2]:
print('as_of 2024-02-20 (Feb未公表・Jan改定前):')
print(as_of(frame, '2024-02-20'))
print('\nas_of 2024-03-20 (Jan改定後・Feb公表後):')
print(as_of(frame, '2024-03-20'))
print('\nlatest (最新改定値):')
print(latest(frame))
print('\n1月分の改定履歴:')
print(revisions(frame, '2024-01-01'))

as_of 2024-02-20 (Feb未公表・Jan改定前):
period_start
2024-01-01    3.0
Name: value, dtype: float64

as_of 2024-03-20 (Jan改定後・Feb公表後):
period_start
2024-01-01    3.2
2024-02-01    2.8
Name: value, dtype: float64

latest (最新改定値):
period_start
2024-01-01    3.2
2024-02-01    2.8
Name: value, dtype: float64

1月分の改定履歴:
  release_date  value  vintage_available
0   2024-02-15    3.0               True
1   2024-03-15    3.2               True


`as_of('2024-02-20')` は 1月分=3.0(初出)のみ。2月分は未公表、1月の改定(03-15)も未来なので**見えない**。これが改定マクロでの未来漏れを防ぐ唯一の門。

## 実データ(任意): FRED/ALFRED の vintage

In [3]:
from irp.macro import FredConnector, as_of, latest
from irp.utils.config import env

if env('FRED_API_KEY'):
    try:
        f = FredConnector().fetch('us_cpi', '2015-01-01', point_in_time=True)
        print('vintage rows:', len(f))
        print('as_of 2020-03-01 (tail):'); print(as_of(f, '2020-03-01').tail(3))
        print('latest (tail):'); print(latest(f).tail(3))
    except Exception as e:
        print('FRED fetch failed:', type(e).__name__, e)
else:
    print('FRED_API_KEY 未設定 — 合成実演のみ。live は .env に鍵を入れて再実行。')

FRED_API_KEY 未設定 — 合成実演のみ。live は .env に鍵を入れて再実行。
